# Faults and ghost nodes in DISU grids

On top of a DISU grid (see `disu_grids.ipynb`), vorflow computes two things a MODFLOW model of a faulted, layered aquifer needs:

1. **Barrier connections**: the connections a fault cuts, as input for the HFB package, including faults that dip through the layers.
2. **Ghost-node corrections**: for connections between cells whose centres are offset, the weights of a ghost head that MODFLOW uses instead of the cell's own head (GNC package). They must not reach across a fault.

This notebook builds both on vorflow meshes, checks them against an exact solution without running MODFLOW, and shows the arrays that go to FloPy. How barrier faces are meshed in 2D is in `barrier_meshing.ipynb`.

In [ ]:
import tempfile
import time
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import LineString, Point, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, disu

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

workspace = Path(tempfile.mkdtemp(prefix="vorflow_faults_"))
domain = box(0, 0, 200, 120)


def mesh(fault=None):
    # A 10 m Voronoi mesh of the domain, with the fault as a barrier line if given.
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id=1, resolution=10)
    if fault is not None:
        cm.add_line(fault, line_id="fault", resolution=6, is_barrier=True)
    mg = MeshGenerator(background_lc=10, conceptual_mesh=cm, verbosity=0)
    mg.generate()
    return VoronoiTessellator(mg, cm).generate()


def plan_of(d, node):
    # The plan polygon of a DISU node.
    row = d.cellmap[d.cellmap.node == node].iloc[0]
    return d.grids_per_layer[row.layer].geometry.iloc[row.icpl]

## 1. A vertical fault

The fault strikes obliquely across the domain. With `add_line(..., is_barrier=True)` the mesh has faces exactly on it. `DisuGrid.barrier_faces(trace)` returns every connection the fault cuts: a horizontal connection is cut when the segment between its two cell centres crosses the trace. On a mesh that does not follow the trace, the same rule picks the faces nearest to it, a zig-zag that still separates the two sides (`on_trace` tells which is which).

In [ ]:
fault = LineString([(70, 0), (110, 120)])
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
for ax, (name, grid) in zip(axes, (("meshed with is_barrier=True", mesh(fault)), ("mesh that ignores the fault", mesh()))):
    x = grid.x.to_numpy()
    d = disu.DisuGrid(grid, 40 - 0.05 * x, [30 - 0.05 * x, 20 - 0.05 * x, 10 - 0.05 * x]).build()
    faces = d.barrier_faces(fault)
    layer0 = [(n, m) for n, m in faces.pairs if d.cellmap.set_index("node").layer[n] == 0]
    grid.boundary.plot(ax=ax, color="0.6", linewidth=0.4)
    for n, m in layer0:
        face = plan_of(d, n).boundary.intersection(plan_of(d, m).boundary)
        ax.plot(*np.array(shapely.get_coordinates(face)).T, color="tab:red", lw=3)
    ax.plot(*fault.xy, color="k", ls="--", lw=1.2)
    ax.set(xlim=(40, 140), ylim=(0, 120), xlabel="x (m)", aspect="equal",
           title=f"{name}\n{len(faces)} barrier connections, {faces.on_trace.sum()} on the trace")
axes[0].set_ylabel("y (m)")
axes[1].set_ylabel("")
fig.legend(handles=[Line2D([], [], color="0.6", lw=0.6, label="cells"),
                    Line2D([], [], color="k", ls="--", label="fault trace"),
                    Line2D([], [], color="tab:red", lw=3, label="faces cut by the fault (layer 1)")],
           loc="lower center", ncol=3, frameon=False)
fig.subplots_adjust(bottom=0.18)
plt.show()

## 2. A dipping fault

A dipping fault needs a different plan mesh in each layer: each layer is meshed with the fault at that layer's mid-depth. Here four flat layers 10 m thick are cut by a fault dipping at 45°, so each layer's trace is 10 m further east (across the strike) than the one above. Give `barrier_faces` one trace per layer (`None` where a layer has no fault). The fault becomes a staircase:

- **walls**: faces on a layer's own trace (horizontal connections);
- **treads**: vertical connections whose two cells meet between the traces of two consecutive layers, inside the polygon joining them (closed at the fault tips). `faces.vertical` marks them.

In [ ]:
normal = np.array([120.0, -40.0]) / np.hypot(120.0, 40.0)   # horizontal normal of the traces
shift = 10.0 / normal[0]                                     # x shift for 10 m across the strike
traces = [LineString([(70 + k * shift, 0), (110 + k * shift, 120)]) for k in range(4)]
grids = [mesh(t) for t in traces]
d = disu.DisuGrid(grids, 40.0, [30.0, 20.0, 10.0, 0.0]).build()
faces = d.barrier_faces(traces)
cells = d.cellmap[d.cellmap.node >= 0].sort_values("node")
layer, icpl = cells.layer.to_numpy(), cells.icpl.to_numpy()
print(f"{len(faces)} barrier connections: {(~faces.vertical).sum()} walls ({faces.on_trace.sum()} on a trace), "
      f"{faces.vertical.sum()} treads; factor {faces.factor.min():.3f} to {faces.factor.max():.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
grids[0].boundary.plot(ax=ax, color="0.6", linewidth=0.4)
treads = [(n, m) for n, m, v in zip(faces.n, faces.m, faces.vertical) if v and {layer[n], layer[m]} == {0, 1}]
gpd.GeoSeries([plan_of(d, n).intersection(plan_of(d, m)) for n, m in treads]).plot(
    ax=ax, facecolor="tab:blue", edgecolor="white", lw=0.3, alpha=0.8)
for n, m, v in zip(faces.n, faces.m, faces.vertical):
    if not v and layer[n] == layer[m] == 0:
        ax.plot(*shapely.get_coordinates(plan_of(d, n).boundary.intersection(plan_of(d, m).boundary)).T, color="tab:red", lw=3)
ax.plot(*traces[1].xy, color="tab:orange", lw=2, ls="--")
ax.legend(handles=[Line2D([], [], color="0.6", lw=0.6, label="layer 1 cells"),
                   Line2D([], [], color="tab:red", lw=3, label="walls: layer 1 faces on its trace"),
                   Line2D([], [], color="tab:orange", lw=2, ls="--", label="trace of layer 2 (10 m below)"),
                   Patch(facecolor="tab:blue", alpha=0.8, label="treads: layer 1 / layer 2 contacts\nbetween the two traces")],
          loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.set(xlim=(40, 160), ylim=(0, 120), xlabel="x (m)", ylabel="y (m)", title="Barrier connections of layer 1 (plan)")
ax.set_aspect("equal")
plt.show()

A section across the strike shows the staircase. Cells are coloured by the side of the fault plane (through the traces at mid-depth) their centre is on; every connection between the two colours is a wall or a tread.

In [ ]:
a = np.array(traces[0].coords[0])
centre = np.array([95.0, 60.0])
line = LineString([centre - 60 * normal, centre + 60 * normal])
props = d.get_gridprops_mf6_disu()
z = (props["top"] + props["bot"]) / 2
xy = cells[["x", "y"]].to_numpy()
beyond = (xy - a) @ normal - (35.0 - z) > 0             # beyond the fault plane


def section_cells(d, line):
    # Active cells cut by a plan line: node, layer, start and end along the line, bottom and top.
    rows = []
    for row in d.cellmap[d.cellmap.node >= 0].itertuples():
        cut = d.grids_per_layer[row.layer].geometry.iloc[row.icpl].intersection(line)
        if not cut.is_empty and cut.length > 0:
            s = [line.project(Point(p)) for p in shapely.get_coordinates(cut)]
            rows.append((row.node, row.layer, min(s), max(s), props["bot"][row.node], props["top"][row.node]))
    return pd.DataFrame(rows, columns=["node", "layer", "s0", "s1", "bot", "top"]).set_index("node")


sec = section_cells(d, line)
fig, ax = plt.subplots(figsize=(13, 4.5))
for node, c in sec.iterrows():
    ax.add_patch(Rectangle((c.s0, c.bot), c.s1 - c.s0, c.top - c.bot, facecolor="#9ecae1" if beyond[node] else "#d8c8a8",
                           edgecolor="0.4", lw=0.4))
for n, m, v in zip(faces.n, faces.m, faces.vertical):
    if n in sec.index and m in sec.index:
        p, q = sec.loc[n], sec.loc[m]
        if v:   # tread: along the shared contact, at the interface
            s0, s1, zc = max(p.s0, q.s0), min(p.s1, q.s1), (p.bot if p.bot > q.bot else q.bot)
            if s1 > s0:
                ax.plot([s0, s1], [zc, zc], color="tab:red", lw=3)
        elif abs(p.s1 - q.s0) < 1e-6 or abs(q.s1 - p.s0) < 1e-6:   # wall: where the two cells meet along the line
            s = p.s1 if abs(p.s1 - q.s0) < 1e-6 else p.s0
            ax.plot([s, s], [max(p.bot, q.bot), min(p.top, q.top)], color="tab:red", lw=3)
zz = np.array([0.0, 40.0])
ax.plot(35.0 - zz - (line.coords[0] - a) @ normal + 0 * zz, zz, color="k", ls="--", lw=1.2)
ax.set(xlim=(0, line.length), ylim=(0, 40), xlabel="distance along the section, across the strike (m)", ylabel="z (m)",
       title="Section across the fault")
ax.legend(handles=[Patch(facecolor="#d8c8a8", edgecolor="0.4", label="cells on the near side"),
                   Patch(facecolor="#9ecae1", edgecolor="0.4", label="cells beyond the fault"),
                   Line2D([], [], color="tab:red", lw=3, label="barrier faces: walls and treads"),
                   Line2D([], [], color="k", ls="--", label="fault plane (45°)")],
          loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.show()

**The face factor.** The staircase has more face area than the fault plane, and its faces are tilted to it. `faces.factor` is |n<sub>fault</sub> · n<sub>face</sub>|, with the dip read from the offset between consecutive traces at their layers' mid-depths (0.707 at 45°); it does not depend on K. The HFB exporters multiply the hydraulic characteristic by it, so that flow normal to the fault meets the plane's resistance on the staircase (checked in section 4). Treads are vertical HFB connections: MODFLOW 6 accepts them from version 6.7; the MODFLOW-USG export refuses them.

## 3. Ghost nodes

The four layers here have different meshes, so the cells joined by a vertical connection are offset in plan. A two-point flux between offset cells is wrong for any flow along the layers. A ghost-node correction replaces the host cell's head, in that connection only, by the head at a ghost point level with the other cell, interpolated from the host's immediate neighbours:

h<sub>ghost</sub> = h<sub>n</sub> + Σ α<sub>j</sub> (h<sub>j</sub> − h<sub>n</sub>)

`estimate_gnc` finds the weights α from the geometry alone. With `barriers=faces.pairs` no neighbour across the fault is used: across a barrier the head jumps, so such a neighbour would bias the ghost head.

`estimate_gnc` is the slow step of this notebook: it solves a small linear program for each end of every offset connection, so its run time grows with the number of offset connections, not the cell count. The cell below prints the time: here about 2.5 ms per offset connection, so a model with 100,000 offset connections needs about four minutes. Geometric weights depend only on the grid, so compute them once per grid; K-aware weights must be recomputed whenever K changes, which matters in a calibration loop.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # unsupported connections are listed in .unsupported
    start = time.perf_counter()
    plain = d.estimate_gnc(connections="all")
    seconds = time.perf_counter() - start
    aware = d.estimate_gnc(connections="all", barriers=faces.pairs)
for name, c in (("without barriers", plain), ("with barriers=faces.pairs", aware)):
    print(f"{name}: {c.corrected_connections} connections corrected ({c.record_count} ghost records), "
          f"{len(c.unsupported)} left without a correction")
offset = plain.corrected_connections + len(plain.unsupported)
print(f"estimate_gnc took {seconds:.1f} s for {d.nodes} nodes, {len(d.links['n'])} connections, "
      f"{offset} of them offset ({1000 * seconds / offset:.1f} ms each)")

Next to the fault, `barriers=` changes the records: a ghost that used a neighbour across the fault is rebuilt from neighbours on its own side, if they can place it. Otherwise the connection gets no ghost record: it keeps its HFB and the two-point flux (section 4 shows why and how much that matters). Below, one connection both ways: the host cell that holds the ghost, the cell it connects to, the ghost point, and the neighbours whose heads it uses (with their weights). With `barriers=` the ghost moved to the other cell of the connection.

In [ ]:
cut = set(map(tuple, np.sort(faces.pairs, axis=1)))
crossing = [r for r in plain.records if any(tuple(sorted((r["n"], j))) in cut for j in r["contributors"])]
rebuilt = {}
for r in aware.records:
    rebuilt.setdefault(r["link"], r)
no_record = {u["link"] for u in aware.unsupported}
print(f"{len(crossing)} ghost records use a neighbour across the fault without barriers=. With it, "
      f"{sum(r['link'] in rebuilt for r in crossing)} are rebuilt on the other side and "
      f"{sum(r['link'] in no_record for r in crossing)} get no ghost record (HFB and two-point flux).")
record = next(r for r in crossing if r["link"] in rebuilt)
pair = shapely.union_all([plan_of(d, record["n"]), plan_of(d, record["m"])]).buffer(15).envelope

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, (name, r) in zip(axes, (("without barriers", record), ("with barriers=faces.pairs", rebuilt[record["link"]]))):
    n, m = r["n"], r["m"]
    box_ = pair
    grids[layer[n]].boundary.plot(ax=ax, color="0.75", linewidth=0.4)
    gpd.GeoSeries([plan_of(d, n)]).plot(ax=ax, facecolor="#fde0b0", edgecolor="k", lw=1.5)
    gpd.GeoSeries([plan_of(d, m)]).boundary.plot(ax=ax, color="tab:purple", lw=2, ls="--")
    ax.plot(*traces[layer[n]].xy, color="tab:red", lw=2.5)
    for j, w in zip(r["contributors"], r["weights"]):
        across = tuple(sorted((n, j))) in cut
        ax.plot(*xy[j], "s", color="tab:red" if across else "tab:green", ms=9)
        ax.annotate(f"{w:.2f}", xy[j], xytext=(6, 6), textcoords="offset points", fontsize=9)
    ax.plot(*xy[n], "ko", ms=5)
    ax.plot(*r["target"][:2], "*", color="tab:blue", ms=15)
    x0, y0, x1, y1 = box_.bounds
    ax.set(xlim=(x0, x1), ylim=(y0, y1), xlabel="x (m)", aspect="equal",
           title=f"{name}\nghost in the layer {layer[n] + 1} cell, {len(r['contributors'])} neighbours used")
axes[0].set_ylabel("y (m)")
axes[1].set_ylabel("")
fig.legend(handles=[Patch(facecolor="#fde0b0", edgecolor="k", label="host cell (holds the ghost)"),
                    Line2D([], [], color="tab:purple", ls="--", lw=2, label="the cell it connects to"),
                    Line2D([], [], color="tab:red", lw=2.5, label="fault trace in the host's layer"),
                    Line2D([], [], marker="*", color="tab:blue", ls="none", ms=12, label="ghost point"),
                    Line2D([], [], marker="s", color="tab:green", ls="none", ms=8, label="neighbour used, same side (weight)"),
                    Line2D([], [], marker="s", color="tab:red", ls="none", ms=8, label="neighbour used, across the fault (weight)")],
           loc="lower center", ncol=3, frameon=False)
fig.subplots_adjust(bottom=0.2)
plt.show()

**K-aware weights.** `method="k_aware"` weights a neighbour by Darcy's law across the face between them, where K changes by at least `contrast` (default 10) across that face; smaller changes are treated as the same material, so smooth fields such as pilot-point interpolations get geometric weights. Here the cells east of x = 150 m have a lower K, by 2× or by 20×; one ghost record whose neighbours straddle that contact:

In [ ]:
east = xy[:, 0] > 150
key = lambda r: (r["n"], r["link"])  # noqa: E731   the host cell and the connection
vertical = next(r for r in aware.records if any(east[j] != east[r["n"]] for j in r["contributors"]))
rows = []
for ratio in (2, 20):
    k = np.where(east, 1.0 / ratio, 1.0)
    # Geometric weights do not depend on K: they are the ones computed above.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        corrections = {"geometric": aware,
                       "k_aware": d.estimate_gnc(k, method="k_aware", connections="all", barriers=faces.pairs)}
    for method, corr in corrections.items():
        r = next(x for x in corr.records if key(x) == key(vertical))
        rows.append({"K ratio": f"{ratio}x", "method": method,
                     "weights": ", ".join(f"{w:.3f}" for w in r["weights"])})
print(pd.DataFrame(rows).to_string(index=False))

At 2× the contact is below the threshold and K-aware gives the geometric weights; at 20× the neighbour across the contact weighs less, because water crosses the lower K. Pass `contrast=` to change the threshold (`contrast=1` treats every difference as a contact).

## 4. Checks without a solver

The exact head field is known for a planar fault in a uniform aquifer (K = 1): the same flux q on both sides, and a head drop q·n / C across the fault, where C is its hydraulic characteristic (0.01 /d here). Putting it into the cells checks what vorflow built, without running MODFLOW:

- **ghost heads**: the head interpolated at each ghost point against the exact head there;
- **flow through the fault**: the two-point flow of every barrier connection, with the HFB series conductance and the ghost-node correction, against the exact flow through that face. The flux here is normal to the fault, the case the staircase represents exactly with the factor.

In [ ]:
C, n3 = 0.01, np.r_[normal, 1.0] / np.sqrt(2)                  # hydraulic characteristic; fault plane normal
q = 0.01 * n3
xyz = np.column_stack((xy, z))
exact = 50.0 - xyz @ q - beyond * (q @ n3) / C


def ghost_errors(corr):
    errors = []
    for r in corr.records:
        h = exact[r["n"]] + np.dot(r["weights"], exact[r["contributors"]] - exact[r["n"]])
        target = np.asarray(r["target"])
        errors.append(abs(h - (50.0 - target @ q - beyond[r["n"]] * (q @ n3) / C)))
    return np.array(errors)


links = d.links
nn, mm = links["n"], links["m"]
area = np.asarray(links["fahl"], float)
face_normal = np.column_stack((np.cos(np.radians(links["angle"])), np.sin(np.radians(links["angle"])), np.zeros(len(nn))))
up = links["ihc"] == 0
face_normal[up] = 0.0
face_normal[up, 2] = np.sign(z[mm[up]] - z[nn[up]])
exact_flow = -(face_normal @ -q) * area


def barrier_flows(factor, corr=None):
    # Flow through every connection: two-point flux with the HFB in series, plus the ghost-node terms.
    conductance = area / (links["cl_n"] + links["cl_m"])
    hfb = np.full(len(nn), np.inf)
    hfb[faces.links] = C * factor
    conductance = 1 / (1 / conductance + 1 / (hfb * area))
    flow = conductance * (exact[nn] - exact[mm])
    for r in (corr.records if corr is not None else []):
        extra = conductance[r["link"]] * np.dot(r["weights"], exact[r["contributors"]] - exact[r["n"]])
        flow[r["link"]] += extra if r["n"] == nn[r["link"]] else -extra
    return flow[faces.links]


results = {
    "ghost heads, GNC": ghost_errors(plain),
    "ghost heads, GNC with barriers=": ghost_errors(aware),
}
flows = {
    "HFB without the factor": barrier_flows(np.ones(len(faces))),
    "HFB with the factor": barrier_flows(faces.factor),
    "HFB with the factor + GNC with barriers=": barrier_flows(faces.factor, aware),
}
# Barrier connections that GNC with barriers= cannot correct: they keep the HFB and the two-point flux.
no_ghost = np.isin(faces.links, [u["link"] for u in aware.unsupported])
toward_far_side = np.where(beyond[mm[faces.links]], 1.0, -1.0)
exact_barrier = exact_flow[faces.links]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
labels = [k.replace(", ", ",\n") for k in results]
bars = axes[0].bar(labels, [v.max() for v in results.values()], color=["tab:red", "tab:green"])
for bar, v in zip(bars, (v.max() for v in results.values())):
    axes[0].annotate(f"{v:.2g} m", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
axes[0].set(ylabel="largest ghost-head error (m)", ylim=(0, 1.15 * max(v.max() for v in results.values())),
            title=f"Ghost heads ({(q @ n3) / C:.1f} m head drop across the fault)")
x0 = np.arange(len(flows))
for offset, group, colour, label in ((-0.2, ~no_ghost, "tab:green", f"with a ghost record or aligned ({(~no_ghost).sum()})"),
                                     (0.2, no_ghost, "tab:orange", f"no ghost record, HFB only ({no_ghost.sum()})")):
    largest = [100 * np.max(np.abs(f[group] - exact_barrier[group]) / np.abs(exact_barrier[group])) for f in flows.values()]
    axes[1].bar(x0 + offset, np.maximum(largest, 1e-14), width=0.38, color=colour, label=label)
    for x, v in zip(x0 + offset, largest):
        axes[1].annotate(f"{v:.2g} %", (x, max(v, 1e-14)), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
axes[1].set_xticks(x0, [k.replace(" + ", "\n+ ") for k in flows], fontsize=9)
axes[1].set(yscale="log", ylim=(1e-14, 1e4), ylabel="largest error of one barrier connection (%)",
            title="Flow through the barrier connections")
axes[1].legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()
for name, flow in flows.items():
    total = 100 * ((toward_far_side * flow).sum() / (toward_far_side * exact_barrier).sum() - 1)
    print(f"{name}: total flow through the fault {total:+.2f} %")

Without `barriers=`, ghost points beside the fault average heads across the head drop. With it, every ghost head is exact. The factor makes the barrier flows right where cells are aligned, and the ghost-node correction does the rest where they are offset (green).

Some treads get no ghost record (orange). Their two cells touch only in a thin sliver along the trace, so each ghost point, at the centre of that sliver, sits at the fault edge of its cell. A ghost head is a weighted average of the cell and its neighbours, with weights that are not negative and add up to at most 1, so the ghost point must lie inside the hull of their centres. On the cell's own side the neighbours all lie away from the fault and the hull stops short of it; only neighbours across the fault would reach it, and `barriers=` leaves those out because the head jumps there. These connections still have the HFB, so the barrier has no hole. They keep MODFLOW's two-point flux, which errs by a few percent here, but they carry little water, as the total flow through the fault shows. They are listed in `.unsupported`.

## 5. The arrays for MODFLOW

The barrier connections and the ghost records come as FloPy keyword arguments. `hydchr` is the fault's hydraulic characteristic (conductivity divided by width); the exporters multiply it by the factor.

In [ ]:
hfb = faces.get_gridprops_mf6_hfb(C)
print("ModflowGwfhfb:", {"maxhfb": hfb["maxhfb"], "first records": hfb["stress_period_data"][0][:3]})
gnc = aware.get_gridprops_mf6_gnc()
print("ModflowGwfgnc:", {key: (f"{len(v)} records" if key == "gncdata" else v) for key, v in gnc.items()})
usg = aware.get_gridprops_mfusg_gnc()
print("MfUsgGnc:", {key: (f"{len(v)} records" if key == "gncdata" else v) for key, v in usg.items()})
print("double-precision USG GNC file:", aware.write_mfusg_gnc(workspace / "model.gnc"))
try:
    faces.get_gridprops_mfusg_hfb(C)
except ValueError as error:
    print("MODFLOW-USG HFB:", error)

```python
flopy.mf6.ModflowGwfdisu(gwf, **d.get_gridprops_mf6_disu())
flopy.mf6.ModflowGwfhfb(gwf, **faces.get_gridprops_mf6_hfb(hydchr))      # MODFLOW 6 >= 6.7 for the treads
flopy.mf6.ModflowGwfgnc(gwf, **aware.get_gridprops_mf6_gnc())           # without XT3D in the same model
```

**Guidance:**

- **Keep the factor.** The exporters apply it; give the hydraulic characteristic (>= 0), not a conductance multiplier.
- **Pass `barriers=` to `estimate_gnc`** whenever the model has a barrier.
- **GNC or XT3D, not both.** Beside offset cells, MODFLOW 6's XT3D treatment of HFB loses accuracy, so GNC with `barriers=` is the better choice next to faults.
- **Flow along a dipping fault or unit** sees the staircase: keep Δx·tanθ/b at or below about 5 % (see `disu_grids.ipynb`).
- **K-aware weights** depend on K: recompute them whenever K changes.